# Multi-Contract Model Runs (AG, AL, AU, BB, BU)This notebook trains four models (Logistic Regression, SVM, Random Forest, XGBoost if available) on five contracts (AG, AL, AU, BB, BU) using the shared Python modules in `esl_project`. Each model has its own training cell followed by a visualization cell. Figures are saved with timestamped names into per-contract folders.

In [ ]:
from pathlib import Path
from datetime import datetime
import importlib.util

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import matplotlib.dates as mdates
import numpy as np
from sklearn.metrics import (
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
    brier_score_loss,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from esl_project.data_utils import DataLoadConfig, list_csv_files
from esl_project.pipelines import ContractRunConfig, run_single_contract, short_contract_tag

plt.style.use("seaborn-v0_8-whitegrid")

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = Path("outputs_parallel") / f"ESL_run_{RUN_TIMESTAMP}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("2005年__20250905")
CONTRACT_CODES = ["AG", "AL", "AU", "BB", "BU"]
print(f"Run timestamp: {RUN_TIMESTAMP}")
print(f"Output root: {OUTPUT_ROOT.resolve()}")


In [ ]:
all_files = list_csv_files(DATA_DIR)contract_paths = {}for code in CONTRACT_CODES:    matches = [p for p in all_files if p.name.startswith(f"{code}_")]    if not matches:        print(f"[Warn] No CSV found for {code} under {DATA_DIR}")        continue    contract_paths[code] = matches[0]if not contract_paths:    raise FileNotFoundError("None of the requested contract CSVs were found.")else:    for code, path in contract_paths.items():        print(f"Using {code}: {path.name}")load_cfg = DataLoadConfig(    data_dir=DATA_DIR,    max_files=None,    nrows_per_file=None,    start_date="2014-01-01",    end_date="2020-12-31",)

In [ ]:

def build_binary_targets(pred_df: pd.DataFrame):
    "Prepare binary targets/probabilities for class 1 diagnostics."
    if pred_df is None or pred_df.empty:
        return None, None, None
    df = pred_df.copy()
    df["index"] = pd.to_datetime(df["index"])
    y_true = (df["true"] == 1).astype(int)
    prob_like = df["prob_1"].fillna(df.get("score_1", np.nan)).fillna(0.0)
    return df, y_true, prob_like


def plot_confusion_and_rolling(model_label: str, tag: str, results: dict):
    "Plot confusion heatmap and rolling metrics, then save with timestamped names."
    fig_dir = results["fig_dir"]
    fig_dir.mkdir(parents=True, exist_ok=True)
    conf = pd.DataFrame(results["confusion"], index=[-1, 0, 1], columns=[-1, 0, 1])
    metrics_df = results["rolling_metrics"]

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(conf, annot=True, fmt="g", cmap="Blues", ax=ax)
    ax.set_title(f"{tag} {model_label} Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    fig.savefig(fig_dir / f"{tag}_{model_label}_confusion_{RUN_TIMESTAMP}.png", dpi=150, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4))
    if metrics_df is not None and not metrics_df.empty:
        x_vals = pd.to_datetime(metrics_df["test_end"])
        ax.plot(x_vals, metrics_df["accuracy"], marker="o", label="Accuracy")
        ax.plot(x_vals, metrics_df["f1_macro"], marker="o", label="Macro F1")
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        fig.autofmt_xdate()
    ax.set_title(f"{tag} {model_label} Rolling Performance")
    ax.set_xlabel("Test month end")
    ax.set_ylabel("Score")
    ax.legend()
    fig.savefig(fig_dir / f"{tag}_{model_label}_rolling_{RUN_TIMESTAMP}.png", dpi=150, bbox_inches="tight")
    plt.show()


def plot_probability_diagnostics(model_label: str, tag: str, results: dict):
    df, y_true, score = build_binary_targets(results.get("predictions"))
    if y_true is None:
        print(f"[{model_label}] No predictions available for diagnostics.")
        return
    fig_dir = results["fig_dir"]
    fig_dir.mkdir(parents=True, exist_ok=True)

    fpr, tpr, _ = roc_curve(y_true, score)
    roc_auc = auc(fpr, tpr)
    precision, recall, _ = precision_recall_curve(y_true, score)
    ap = average_precision_score(y_true, score)
    brier = brier_score_loss(y_true, score)
    bal_acc = balanced_accuracy_score(y_true, (score >= 0.5).astype(int))

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
    axes[0].set_title(f"{tag} {model_label} ROC")
    axes[0].set_xlabel("FPR")
    axes[0].set_ylabel("TPR")
    axes[0].legend()

    axes[1].plot(recall, precision, label=f"AP={ap:.3f}")
    axes[1].set_title(f"{tag} {model_label} PR Curve")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(fig_dir / f"{tag}_{model_label}_roc_pr_{RUN_TIMESTAMP}.png", dpi=150, bbox_inches="tight")
    plt.show()

    bins = np.linspace(0, 1, 11)
    bin_ids = np.digitize(score, bins) - 1
    calib = pd.DataFrame({"prob": score, "y": y_true, "bin": bin_ids})
    calib_grp = calib.groupby("bin").agg(prob_mean=("prob", "mean"), true_rate=("y", "mean")).dropna()

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(calib_grp["prob_mean"], calib_grp["true_rate"], marker="o", label="Observed")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect")
    ax.set_title(f"{tag} {model_label} Calibration (Brier={brier:.3f})")
    ax.set_xlabel("Predicted prob (class 1)")
    ax.set_ylabel("Observed freq")
    ax.legend()
    fig.savefig(fig_dir / f"{tag}_{model_label}_calibration_{RUN_TIMESTAMP}.png", dpi=150, bbox_inches="tight")
    plt.show()

    resid = y_true - score
    lags = range(1, 13)
    acf_vals = [pd.Series(resid).autocorr(lag=lag) for lag in lags]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(df["index"], resid, color="steelblue")
    axes[0].set_title("Residuals over time")
    axes[0].set_xlabel("Time")
    axes[0].set_ylabel("y - p")
    fig.autofmt_xdate()

    axes[1].hist(resid, bins=30, color="teal", alpha=0.7, edgecolor="black")
    axes[1].set_title("Residual distribution")

    axes[2].bar(list(lags), acf_vals, color="darkorange")
    axes[2].set_title("Residual ACF (lags 1-12)")
    axes[2].set_xlabel("Lag")
    axes[2].set_ylabel("Autocorr")
    fig.tight_layout()
    fig.savefig(fig_dir / f"{tag}_{model_label}_residuals_{RUN_TIMESTAMP}.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"[{model_label}] AUC={roc_auc:.3f}, AP={ap:.3f}, Brier={brier:.3f}, Balanced Acc(0.5 thr)={bal_acc:.3f}")


def plot_threshold_sweep(model_label: str, tag: str, results: dict):
    df, y_true, score = build_binary_targets(results.get("predictions"))
    if y_true is None:
        return
    fig_dir = results["fig_dir"]
    thresholds = np.linspace(0.05, 0.95, 10)
    rows = []
    for thr in thresholds:
        y_hat = (score >= thr).astype(int)
        rows.append(
            {
                "threshold": thr,
                "precision": precision_score(y_true, y_hat, zero_division=0),
                "recall": recall_score(y_true, y_hat, zero_division=0),
                "f1": f1_score(y_true, y_hat, zero_division=0),
                "balanced_accuracy": balanced_accuracy_score(y_true, y_hat),
            }
        )
    thr_df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(thr_df["threshold"], thr_df["f1"], marker="o", label="F1")
    ax.plot(thr_df["threshold"], thr_df["precision"], marker="o", label="Precision")
    ax.plot(thr_df["threshold"], thr_df["recall"], marker="o", label="Recall")
    ax.plot(thr_df["threshold"], thr_df["balanced_accuracy"], marker="o", label="Balanced Acc")
    ax.set_title(f"{tag} {model_label} Threshold Sweep")
    ax.set_xlabel("Probability threshold for class 1")
    ax.set_ylabel("Score")
    ax.legend()
    fig.savefig(fig_dir / f"{tag}_{model_label}_thresholds_{RUN_TIMESTAMP}.png", dpi=150, bbox_inches="tight")
    plt.show()


def plot_model_diagnostics(model_label: str, tag: str, results: dict):
    plot_confusion_and_rolling(model_label, tag, results)
    plot_probability_diagnostics(model_label, tag, results)
    plot_threshold_sweep(model_label, tag, results)


In [ ]:
logit_cfg = ContractRunConfig(    candidate_C=[0.01, 0.1],    model_name="logit",    train_months=12,    test_months=1,    downsample_every=2,)logit_results = {}for code, path in contract_paths.items():    tag = short_contract_tag(path.stem)    res = run_single_contract(path, OUTPUT_ROOT, logit_cfg, load_cfg)    logit_results[code] = res    print(f"[Logit][{code}] Best params: {res.get('best_params')}")    print(f"[Logit][{code}] Summary head:{res.get('summary').head()}")

In [ ]:
for code, res in logit_results.items():    plot_model_diagnostics("logit", short_contract_tag(res.get("contract", code)), res)

In [ ]:
svm_cfg = ContractRunConfig(    candidate_C=[0.5],    model_name="svm",    param_grid=[{"C": 0.5, "tol": 1e-3, "max_iter": 1000}],    train_months=12,    test_months=1,    downsample_every=5,)svm_results = {}for code, path in contract_paths.items():    tag = short_contract_tag(path.stem)    res = run_single_contract(path, OUTPUT_ROOT, svm_cfg, load_cfg)    svm_results[code] = res    print(f"[SVM][{code}] Best params: {res.get('best_params')}")    print(f"[SVM][{code}] Summary head:{res.get('summary').head()}")

In [ ]:
for code, res in svm_results.items():    plot_model_diagnostics("svm", short_contract_tag(res.get("contract", code)), res)

In [ ]:
rf_cfg = ContractRunConfig(    candidate_C=[0.1],    model_name="rf",    param_grid=[{"n_estimators": 120, "max_depth": 8, "max_features": "sqrt", "n_jobs": -1}],    train_months=12,    test_months=1,    downsample_every=5,)rf_results = {}for code, path in contract_paths.items():    tag = short_contract_tag(path.stem)    res = run_single_contract(path, OUTPUT_ROOT, rf_cfg, load_cfg)    rf_results[code] = res    print(f"[RF][{code}] Best params: {res.get('best_params')}")    print(f"[RF][{code}] Summary head:{res.get('summary').head()}")

In [ ]:
for code, res in rf_results.items():    plot_model_diagnostics("rf", short_contract_tag(res.get("contract", code)), res)

In [ ]:
xgb_available = importlib.util.find_spec("xgboost") is not Noneif not xgb_available:    print("[XGB] xgboost is not installed; skipping this run.")    xgb_results = Noneelse:    import numpy as np    import esl_project.model_loader as ml    from esl_project.models.xgboost_model import train_xgboost as _train_xgb    _orig_train_model = ml.train_model    class XGBLabelWrapper:        def __init__(self, base_model, inv_map):            self.model = base_model            self.inv_map = inv_map            self._order = [-1, 0, 1]        def predict(self, X):            raw = self.model.predict(X)            return np.array([self.inv_map[int(v)] for v in raw])        def predict_proba(self, X):            base = self.model.predict_proba(X)            out = np.zeros((base.shape[0], len(self._order)))            mapping = {0: -1, 1: 0, 2: 1}            for src_lbl, tgt_lbl in mapping.items():                tgt_idx = self._order.index(tgt_lbl)                out[:, tgt_idx] = base[:, src_lbl]            return out        def __getattr__(self, name):            return getattr(self.model, name)    def patched_train_model(X_train, y_train, config):        if config.model_name.lower() == "xgb":            label_map = {-1: 0, 0: 1, 1: 2}            inv_map = {v: k for k, v in label_map.items()}            y_shift = np.array([label_map[int(v)] for v in y_train])            base = _train_xgb(X_train, y_shift, **config.params)            return XGBLabelWrapper(base, inv_map)        return _orig_train_model(X_train, y_train, config)    ml.train_model = patched_train_model    xgb_cfg = ContractRunConfig(        candidate_C=[0.1],        model_name="xgb",        param_grid=[{"n_estimators": 120, "learning_rate": 0.1, "max_depth": 4, "subsample": 0.8, "colsample_bytree": 0.8, "n_jobs": -1}],        train_months=12,        test_months=1,        downsample_every=5,    )    xgb_results = {}    for code, path in contract_paths.items():        tag = short_contract_tag(path.stem)        res = run_single_contract(path, OUTPUT_ROOT, xgb_cfg, load_cfg)        xgb_results[code] = res        print(f"[XGB][{code}] Best params: {res.get('best_params')}")        print(f"[XGB][{code}] Summary head:{res.get('summary').head()}")

In [ ]:
if xgb_available and xgb_results is not None:    for code, res in xgb_results.items():        plot_model_diagnostics("xgb", short_contract_tag(res.get("contract", code)), res)else:    print("[XGB] Visualization skipped because training did not run.")